#### Import Libraries:

In [1]:
import pandas as pd
import numpy as np

#### Load Dataset:

In [2]:
df = pd.read_csv(r"C:\Users\user\Downloads\OnlineRetail.csv", encoding='latin1')
print(df.shape)
print(df.head())

(541909, 8)
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

      InvoiceDate  UnitPrice  CustomerID         Country  
0  12/1/2010 8:26       2.55     17850.0  United Kingdom  
1  12/1/2010 8:26       3.39     17850.0  United Kingdom  
2  12/1/2010 8:26       2.75     17850.0  United Kingdom  
3  12/1/2010 8:26       3.39     17850.0  United Kingdom  
4  12/1/2010 8:26       3.39     17850.0  United Kingdom  


##### Clean Data:

In [3]:
# Remove missing CustomerIDs
df.dropna(subset=['CustomerID'], inplace=True)

# Remove cancellations (InvoiceNo starting with C)
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove negative or zero quantity and price
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]

print(df.shape)

(397884, 8)


#### Create Revenue Column:

In [4]:
df['Revenue'] = df['Quantity'] * df['UnitPrice']
print(df[['CustomerID','Quantity','UnitPrice','Revenue']].head())

   CustomerID  Quantity  UnitPrice  Revenue
0     17850.0         6       2.55    15.30
1     17850.0         6       3.39    20.34
2     17850.0         8       2.75    22.00
3     17850.0         6       3.39    20.34
4     17850.0         6       3.39    20.34


#### Convert Date Column:

In [5]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df.dtypes)

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID            float64
Country                object
Revenue               float64
dtype: object


#### Save Cleaned File:

In [6]:
df.to_csv('OnlineRetail_Cleaned.csv', index=False)
print("Saved successfully")

Saved successfully


#### Set Reference Date:

In [7]:
import datetime
reference_date = df['InvoiceDate'].max() + datetime.timedelta(days=1)
print(reference_date)

2011-12-10 12:50:00


#### Calculate RFM Values:

In [8]:
rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary  = ('Revenue', 'sum')
).reset_index()

print(rfm.shape)
print(rfm.head())

(4338, 4)
   CustomerID  Recency  Frequency  Monetary
0     12346.0      326          1  77183.60
1     12347.0        2          7   4310.00
2     12348.0       75          4   1797.24
3     12349.0       19          1   1757.55
4     12350.0      310          1    334.40


#### Score Each Customer 1 to 5:

In [9]:
rfm['R_Score'] = pd.qcut(rfm['Recency'],   5, labels=[5,4,3,2,1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'],  5, labels=[1,2,3,4,5])

print(rfm.head())

   CustomerID  Recency  Frequency  Monetary R_Score F_Score M_Score
0     12346.0      326          1  77183.60       1       1       5
1     12347.0        2          7   4310.00       5       5       5
2     12348.0       75          4   1797.24       2       4       4
3     12349.0       19          1   1757.55       4       1       4
4     12350.0      310          1    334.40       1       1       2


#### Create RFM Segment:

In [10]:
def segment_customer(row):
    r = int(row['R_Score'])
    f = int(row['F_Score'])
    m = int(row['M_Score'])
    
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3:
        return 'Loyal Customer'
    elif r >= 4 and f <= 2:
        return 'New Customer'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost Customer'
    else:
        return 'Potential Loyalist'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
print(rfm['Segment'].value_counts())

Segment
Loyal Customer        998
Champion              962
Lost Customer         824
Potential Loyalist    781
At Risk               454
New Customer          319
Name: count, dtype: int64


####  Save RFM File:

In [11]:
rfm.to_csv('RFM_Segments.csv', index=False)
print("RFM file saved successfully")

RFM file saved successfully


In [12]:
print(rfm['Segment'].value_counts())
print("Total Customers:", len(rfm))

Segment
Loyal Customer        998
Champion              962
Lost Customer         824
Potential Loyalist    781
At Risk               454
New Customer          319
Name: count, dtype: int64
Total Customers: 4338


In [14]:
import pandas as pd

# Read with correct separator
df = pd.read_csv(r"C:\Users\user\Downloads\RFM_Segments.xls", 
                 encoding='latin1', 
                 quotechar='"')

# Split the single column into proper columns
df = df.iloc[:, 0].str.split(',', expand=True)
df.columns = ['CustomerID','Recency','Frequency','Monetary','R_Score','F_Score','M_Score','Segment']

# Fix data types
df['CustomerID'] = df['CustomerID'].astype(float)
df['Recency'] = df['Recency'].astype(int)
df['Frequency'] = df['Frequency'].astype(int)
df['Monetary'] = df['Monetary'].astype(float)

print(df.shape)
print(df.head())

# Save correctly
df.to_csv(r"C:\Users\user\Downloads\RFM_Segments_Fixed.csv", index=False)
print("Saved successfully")

(4338, 8)
   CustomerID  Recency  Frequency  Monetary R_Score F_Score M_Score  \
0     12346.0      326          1  77183.60       1       1       5   
1     12347.0        2          7   4310.00       5       5       5   
2     12348.0       75          4   1797.24       2       4       4   
3     12349.0       19          1   1757.55       4       1       4   
4     12350.0      310          1    334.40       1       1       2   

              Segment  
0  Potential Loyalist  
1            Champion  
2             At Risk  
3        New Customer  
4       Lost Customer  
Saved successfully


In [2]:
import pandas as pd

rfm = pd.read_csv(r"C:\Users\user\Downloads\RFM_Segments_Fixed.csv")

print("Total Customers:", len(rfm))
print("\nSegment Counts:")
print(rfm['Segment'].value_counts())
print("\nRevenue by Segment:")
print(rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False).round(2))
print("\nAvg Monetary by Segment:")
print(rfm.groupby('Segment')['Monetary'].mean().sort_values(ascending=False).round(2))

Total Customers: 4338

Segment Counts:
Segment
Loyal Customer        998
Champion              962
Lost Customer         824
Potential Loyalist    781
At Risk               454
New Customer          319
Name: count, dtype: int64

Revenue by Segment:
Segment
Champion              5809341.07
Loyal Customer        1474127.55
At Risk                742149.95
Potential Loyalist     549851.89
Lost Customer          189770.87
New Customer           146166.57
Name: Monetary, dtype: float64

Avg Monetary by Segment:
Segment
Champion              6038.82
At Risk               1634.69
Loyal Customer        1477.08
Potential Loyalist     704.04
New Customer           458.20
Lost Customer          230.30
Name: Monetary, dtype: float64
